In [3]:
from Src.Data_utils import differencing
from dotenv import load_dotenv
from sqlalchemy import create_engine
import os
import pandas as pd

load_dotenv()
##Connect to sql database
engine = create_engine(
    f"mysql+pymysql://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}:{os.getenv('MYSQL_PORT')}/{os.getenv('MYSQL_DB')}"
)

In [4]:
def feature_engineering():
    """
    USE: Puts I(1) and I(0) variables in different tables and creates correcly rescaled GDP per capita variable.

    Returns
    --------
    Data: pd.DataFrame
        2 dataframes one with the relevant I(1) and the other with relevat I(0) variables.
    """
    data = pd.read_sql("SELECT * FROM GDP_inference_clean", engine)

    data.columns = ["date", "Labor Productivity", "Unemployment Rate", "Federal Funds Rate", "Inflation", "GDP",
                    "Population", "Investment", "Government Spending"] ##Renaming the columns
    data["GDP_per_capita"] = data["GDP"]*1000000 / data["Population"] ##Rescaling GDP so that GDP and population are both in thousands

    exclude = ["date", "Unemployment Rate", "Federal Funds Rate", "GDP", "Population"] ##Excluding I(0) and redundant variables
    data_i1 = data.loc[:, ~data.columns.isin(exclude)]

    re_order_cols = ['Government Spending','Labor Productivity' 'Investment', 'GDP_per_capita', 'Inflation'] ##Reorder columns for choletsky decomposition
    data_i1_reorder = data_i1[re_order_cols]
    exogenous_variables = data[["Unemployment Rate", "Federal Funds Rate"]] ##Selecing exogenous variables for VECM
    return data_i1_reorder,exogenous_variables

feature_engineering()

(     Government Spending  Investment  Labor Productivity  GDP_per_capita  \
 0               1811.988    2243.548                 4.1    33868.250501   
 1               1816.961    2244.682                 3.9    34195.108118   
 2               1843.205    2306.470                 3.5    34663.429332   
 3               1875.154    2367.988                 4.2    35315.242012   
 4               1866.308    2351.223                 2.4    35584.432356   
 ..                   ...         ...                 ...             ...   
 102             7165.663    4392.176                 2.6    86684.791804   
 103             7247.662    4315.564                 1.9    87463.363851   
 104             7313.597    4547.947                 1.2    87982.899600   
 105             7496.339    4382.819                 1.5    89172.411473   
 106             7580.229    4383.186                 1.9    90831.635365   
 
      Inflation  
 0     2.383116  
 1     2.226592  
 2     2.111144  
 3